# Preparación de Datasets
Para empezar la práctica, aquí cargamos lo básico. Usamos el dataset Iris de sklearn como nuestro benchmark (150 datos, 4 rasgos numéricos reales y 3 clases) porque ya viene limpio y nos sirve para comprobar que la lógica del código funciona bien.

También generamos un dataset sintético (300 datos, 4 rasgos y 3 clases) al que le agregamos ruido a propósito: con `flip_y=0.15` se cambia al azar la etiqueta del 15% de los datos, así que las clases se traslapan y no hay fronteras claras. Con esto vemos cómo se comportan los modelos en un escenario que no es perfecto.

Además, dejamos listas las funciones para partir los datos usando Hold-Out (70/30) y K-Fold (con k=5).

In [10]:
import numpy as np
from sklearn.datasets import load_iris, make_classification
from sklearn.metrics import accuracy_score

np.random.seed(42)

def particion_hold_out(X, y):
    indices = np.random.permutation(len(X))
    corte = int(len(X) * 0.7)
    train_idx = indices[:corte]
    test_idx = indices[corte:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def evaluacion_kfold(X, y, k, funcion_modelo):
    indices = np.random.permutation(len(X))
    X_revuelto = X[indices]
    y_revuelto = y[indices]
    
    tamano_fold = len(X) // k
    precisiones = []
    
    for i in range(k):
        inicio = i * tamano_fold
        if i < k - 1:
            fin = inicio + tamano_fold
        else:
            fin = len(X)
            
        X_test = X_revuelto[inicio:fin]
        y_test = y_revuelto[inicio:fin]
        
        X_train = np.concatenate((X_revuelto[:inicio], X_revuelto[fin:]))
        y_train = np.concatenate((y_revuelto[:inicio], y_revuelto[fin:]))
        
        y_pred = funcion_modelo(X_train, y_train, X_test)
        precision = accuracy_score(y_test, y_pred)
        precisiones.append(precision)
        
    return np.mean(precisiones)

dataset_benchmark = load_iris()
X_bench = dataset_benchmark.data
y_bench = dataset_benchmark.target

X_sintetico, y_sintetico = make_classification(
    n_samples=300, 
    n_features=4, 
    n_classes=3, 
    n_informative=4, 
    n_redundant=0, 
    flip_y=0.15, 
    random_state=42
)

print("Dataset: CARGADO; Funciones: CARGADO")

Dataset: CARGADO; Funciones: CARGADO


# Modelo de Distancia Mínima
En esta celda se realizo el modelo desde cero sin usar clasificadores preconstruidos. La lógica que se uso se divide en dos pasos: primero, en la parte de entrenamiento, el algoritmo agrupa los datos según su clase y calcula el promedio matemático de cada grupo para encontrar su "centroide" (el punto central representativo de esa clase). 

Después, en la fase de prueba, toma cada dato nuevo, usa la función matemática de la distancia euclidiana para medir qué tan lejos está de cada centroide, y simplemente lo clasifica asignándole la clase del que le quede a la distancia más corta.

In [11]:
def distancia_euclidiana(p1, p2):
    suma = 0.0
    for i in range(len(p1)):
        suma += (p1[i] - p2[i]) ** 2
    return suma ** 0.5

def modelo_distancia_minima(X_train, y_train, X_test):
    clases = np.unique(y_train)
    centroides = []
    
    for c in clases:
        puntos = X_train[y_train == c]
        centroide = np.mean(puntos, axis=0)
        centroides.append((c, centroide))
        
    predicciones = []
    for punto in X_test:
        dist_min = float('inf')
        clase_elegida = -1
        for c, centroide in centroides:
            dist = distancia_euclidiana(punto, centroide)
            if dist < dist_min:
                dist_min = dist
                clase_elegida = c
        predicciones.append(clase_elegida)
        
    return np.array(predicciones)

print("Algoritmo de Distancia Minima")

Algoritmo de Distancia Minima


# Modelo 1-NN (Vecino Más Cercano)
A diferencia de Distancia Mínima, el 1-NN no saca promedios. Lo que hace el algoritmo es tomar el punto de prueba y medir la distancia contra absolutamente todos los puntos que tenemos en el entrenamiento. Al final, se queda con la clase del punto que esté más pegadito a él.

In [12]:
def modelo_1nn(X_train, y_train, X_test):
    predicciones = []
    for punto_test in X_test:
        dist_min = float('inf')
        clase_elegida = -1
        for i in range(len(X_train)):
            dist = distancia_euclidiana(punto_test, X_train[i])
            if dist < dist_min:
                dist_min = dist
                clase_elegida = y_train[i]
        predicciones.append(clase_elegida)
    return np.array(predicciones)

print("Algoritmo 1-NN cargado en memoria.")

Algoritmo 1-NN cargado en memoria.


# Modelo K-NN (K-Nearest Neighbors)
K-NN es una extensión directa del algoritmo 1-NN. En lugar de clasificar un punto nuevo basándose únicamente en su vecino más próximo, K-NN busca los $k$ puntos de entrenamiento que estén a menor distancia (generalmente usando la distancia euclidiana) del punto de prueba. Una vez que encuentra a estos $k$ vecinos, realiza una "votación"; la clase más frecuente entre ellos será la que se le asigne al nuevo dato.

In [13]:
from collections import Counter

def modelo_knn(X_train, y_train, X_test, k=3):
    predicciones = []
    
    for punto_test in X_test:
        # 1. Calculamos la distancia del punto de prueba contra todo el set de entrenamiento
        distancias = []
        for i in range(len(X_train)):
            dist = distancia_euclidiana(punto_test, X_train[i])
            distancias.append((dist, y_train[i]))
        
        # 2. Ordenamos las distancias de menor a mayor
        distancias.sort(key=lambda x: x[0])
        
        # 3. Nos quedamos solo con las clases de los 'k' vecinos más cercanos
        k_vecinos = [clase for _, clase in distancias[:k]]
        
        # 4. Hacemos la votación para encontrar la clase más común
        clase_ganadora = Counter(k_vecinos).most_common(1)[0][0]
        predicciones.append(clase_ganadora)
        
    return np.array(predicciones)

# Función wrapper para usar k=3 fácilmente en nuestra validación
def knn_k3(X_train, y_train, X_test):
    return modelo_knn(X_train, y_train, X_test, k=3)

print("Algoritmo K-NN cargado en memoria.")

Algoritmo K-NN cargado en memoria.


# Naïve Bayes
Naïve Bayes es un clasificador probabilístico basado en el Teorema de Bayes. Calcula la probabilidad de que un punto de prueba pertenezca a cada clase específica, basándose en la frecuencia o distribución de los rasgos (features) en el set de entrenamiento. La parte "Naïve" (ingenua) viene de que asume matemáticamente que todos los rasgos son completamente independientes entre sí (por ejemplo, asume que el ancho del sépalo de una flor no tiene relación con el largo del pétalo, aunque en la realidad sí la tengan). Al final, asigna la clase que obtuvo la probabilidad más alta.

In [14]:
from sklearn.naive_bayes import GaussianNB

def modelo_nb(X_train, y_train, X_test):
    # Inicializamos el clasificador Gaussiano (ideal para datos continuos/reales)
    clf = GaussianNB()
    
    # Entrenamos el modelo con los datos
    clf.fit(X_train, y_train)
    
    # Predecimos las clases para el set de prueba
    return clf.predict(X_test)

print("Algoritmo Naïve Bayes cargado mediante scikit-learn.")

Algoritmo Naïve Bayes cargado mediante scikit-learn.


# SVM (Máquina de Soporte Vectorial)
SVM busca la frontera que mejor separa las clases, es decir, la que deja el mayor margen posible entre los puntos de clases distintas. Con el kernel RBF (el que trae por defecto) puede hacer fronteras curvas, no solo rectas. Como es sensible a la escala de los rasgos, estandarizamos los datos dentro de la función, ajustando el escalador solo con los datos de entrenamiento para no filtrar información del conjunto de prueba.

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def modelo_svm(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_esc = scaler.fit_transform(X_train)   # se ajusta solo con train
    X_test_esc = scaler.transform(X_test)

    clf = SVC()                                   # parámetros por defecto
    clf.fit(X_train_esc, y_train)
    return clf.predict(X_test_esc)

print("SVM cargado mediante scikit-learn.")

SVM cargado mediante scikit-learn.


# MLP (Perceptrón Multicapa)
El MLP es una red neuronal con una o más capas ocultas. Aprende ajustando sus pesos poco a poco para reducir el error de clasificación. Igual que con SVM, estandarizamos los rasgos antes de entrenar. Con los parámetros por defecto usa una capa oculta de 100 neuronas y un máximo de 200 iteraciones, por lo que en algunos casos puede avisar que no terminó de converger; fijamos la semilla para que los resultados se puedan repetir.

In [16]:
from sklearn.neural_network import MLPClassifier

def modelo_mlp(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_esc = scaler.fit_transform(X_train)
    X_test_esc = scaler.transform(X_test)

    clf = MLPClassifier(random_state=42)          # por defecto, con semilla fija
    clf.fit(X_train_esc, y_train)
    return clf.predict(X_test_esc)

print("MLP cargado mediante scikit-learn.")

MLP cargado mediante scikit-learn.


# Comparativa de los seis clasificadores
Evaluamos los seis modelos en los dos datasets con Hold-Out (70/30) y K-Fold (k=5). Para que la comparación sea justa, todos usan la misma partición Hold-Out y los mismos folds, fijando la semilla antes de cada evaluación.

In [17]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore")   # quita los ConvergenceWarning del MLP

modelos = {
    "Distancia Minima": modelo_distancia_minima,
    "1-NN": modelo_1nn,
    "K-NN (k=3)": knn_k3,
    "Naive Bayes": modelo_nb,
    "SVM (default)": modelo_svm,
    "MLP (default)": modelo_mlp,
}

datasets = {
    "Benchmark (Iris)": (X_bench, y_bench),
    "Sintetico (Con Ruido)": (X_sintetico, y_sintetico),
}

filas = []
for nombre_ds, (X, y) in datasets.items():
    # Misma partición Hold Out para todos los modelos
    np.random.seed(42)
    X_tr, X_te, y_tr, y_te = particion_hold_out(X, y)

    for nombre_m, f in modelos.items():
        acc_ho = accuracy_score(y_te, f(X_tr, y_tr, X_te))

        # Misma semilla => mismos folds para todos los modelos
        np.random.seed(42)
        acc_kf = evaluacion_kfold(X, y, 5, f)

        filas.append({
            "Clasificador": nombre_m,
            "Dataset": nombre_ds,
            "Hold-Out (70/30)": round(acc_ho, 4),
            "K-Fold (k=5)": round(acc_kf, 4),
        })

tabla_final = pd.DataFrame(filas)
tabla_final

,Clasificador,Dataset,Hold-Out (70/30),K-Fold (k=5)
0,Distancia Minima,Benchmark (Iris),0.8889,0.9200
1,1-NN,Benchmark (Iris),0.9778,0.9600
2,K-NN (k=3),Benchmark (Iris),0.9778,0.9667
3,Naive Bayes,Benchmark (Iris),0.9556,0.9600
4,SVM (default),Benchmark (Iris),0.9778,0.9667
5,MLP (default),Benchmark (Iris),0.9333,0.9467
6,Distancia Minima,Sintetico (Con Ruido),0.6000,0.5667
7,1-NN,Sintetico (Con Ruido),0.6667,0.6567
8,K-NN (k=3),Sintetico (Con Ruido),0.7222,0.7233
9,Naive Bayes,Sintetico (Con Ruido),0.6111,0.6233


# Pruebas con parámetros distintos al default (K-NN, SVM y MLP)
Para K-NN probamos varios valores de k; para SVM y MLP usamos una búsqueda en rejilla. La búsqueda se hace solo con el 70% de entrenamiento y el 30% de prueba se usa al final para comparar contra el default. El análisis de los resultados está después de la tabla de complejidad.

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

# ---------- K-NN (desde cero): buscamos la mejor k con K-Fold sobre train ----------
valores_k = [1, 3, 5, 7, 9, 11, 15, 21]

def buscar_mejor_k(X_tr, y_tr):
    mejor_k, mejor_acc = None, -1
    for k in valores_k:
        np.random.seed(42)
        acc = evaluacion_kfold(X_tr, y_tr, 5,
                               lambda a, b, c: modelo_knn(a, b, c, k=k))
        if acc > mejor_acc:
            mejor_k, mejor_acc = k, acc
    return mejor_k, mejor_acc

# ---------- Rejillas de parámetros para SVM y MLP ----------
grid_svm = {
    "svc__kernel": ["linear", "rbf", "poly"],
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.01, 0.1, 1],
}
grid_mlp = {
    "mlpclassifier__hidden_layer_sizes": [(5,), (10,), (50,), (100,), (50, 50)],
    "mlpclassifier__activation": ["relu", "tanh"],
    "mlpclassifier__alpha": [0.0001, 0.01, 1],
}

filas_ajuste = []
for nombre_ds, (X, y) in datasets.items():
    # Misma partición que en la tabla final
    np.random.seed(42)
    X_tr, X_te, y_tr, y_te = particion_hold_out(X, y)

    # --- K-NN ---
    k_opt, cv_knn = buscar_mejor_k(X_tr, y_tr)
    acc_def = accuracy_score(y_te, knn_k3(X_tr, y_tr, X_te))
    acc_aj = accuracy_score(y_te, modelo_knn(X_tr, y_tr, X_te, k=k_opt))
    filas_ajuste.append([nombre_ds, "K-NN", "k=3", f"k={k_opt}",
                         round(cv_knn, 4), round(acc_def, 4), round(acc_aj, 4)])

    # --- SVM ---
    gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grid_svm, cv=5)
    gs.fit(X_tr, y_tr)
    acc_def = accuracy_score(y_te, modelo_svm(X_tr, y_tr, X_te))
    acc_aj = accuracy_score(y_te, gs.predict(X_te))
    filas_ajuste.append([nombre_ds, "SVM", "default", str(gs.best_params_),
                         round(gs.best_score_, 4), round(acc_def, 4), round(acc_aj, 4)])

    # --- MLP ---
    gm = GridSearchCV(
        make_pipeline(StandardScaler(), MLPClassifier(max_iter=2000, random_state=42)),
        grid_mlp, cv=5)
    gm.fit(X_tr, y_tr)
    acc_def = accuracy_score(y_te, modelo_mlp(X_tr, y_tr, X_te))
    acc_aj = accuracy_score(y_te, gm.predict(X_te))
    filas_ajuste.append([nombre_ds, "MLP", "default", str(gm.best_params_),
                         round(gm.best_score_, 4), round(acc_def, 4), round(acc_aj, 4)])

tabla_ajuste = pd.DataFrame(
    filas_ajuste,
    columns=["Dataset", "Modelo", "Parámetros default", "Mejores parámetros",
             "CV en train (mejor)", "Test default", "Test ajustado"])
pd.set_option("display.max_colwidth", None)
tabla_ajuste

,Dataset,Modelo,Parámetros default,Mejores parámetros,CV en train (mejor),Test default,Test ajustado
0,Benchmark (Iris),K-NN,k=3,k=7,0.9714,0.9778,0.9778
1,Benchmark (Iris),SVM,default,"{'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}",0.9524,0.9778,0.9778
2,Benchmark (Iris),MLP,default,"{'mlpclassifier__activation': 'relu', 'mlpclassifier__alpha': 0.0001, 'mlpclassifier__hidden_layer_sizes': (10,)}",0.9619,0.9333,0.9556
3,Sintetico (Con Ruido),K-NN,k=3,k=5,0.7190,0.7222,0.7222
4,Sintetico (Con Ruido),SVM,default,"{'svc__C': 1, 'svc__gamma': 1, 'svc__kernel': 'rbf'}",0.7476,0.7444,0.7333
5,Sintetico (Con Ruido),MLP,default,"{'mlpclassifier__activation': 'relu', 'mlpclassifier__alpha': 1, 'mlpclassifier__hidden_layer_sizes': (100,)}",0.7619,0.7444,0.6000


# Análisis de complejidad temporal

Usamos estas variables: **n** = datos de entrenamiento, **d** = número de rasgos, **c** = número de clases, **k** = vecinos y **m** = datos a clasificar.

| Clasificador | Entrenamiento | Predicción (por punto) | Comentario |
|---|---|---|---|
| Distancia Mínima | O(n·d) | O(c·d) | Entrenar es sacar un promedio por clase; clasificar solo compara contra c centroides. |
| 1-NN | O(1) (solo guarda los datos) | O(n·d) | No aprende nada: todo el trabajo se hace al predecir, comparando contra los n puntos. |
| K-NN | O(1) | O(n·d + n·log n) | Igual que 1-NN, pero en nuestra versión además se ordenan las n distancias. Con un heap bajaría a O(n·d + n·log k). |
| Naïve Bayes (Gaussiano) | O(n·d) | O(c·d) | Solo calcula media y varianza por clase y rasgo; es de los más rápidos. |
| SVM (SVC) | entre O(n²·d) y O(n³·d) aprox. | O(sv·d) | Entrenar es lo más caro; sv son los vectores de soporte. Con kernel RBF o polinomial cada comparación cuesta un poco más. |
| MLP | O(e·n·W) | O(W) | e son las épocas y W el número de pesos (con una capa oculta de h neuronas, W ≈ d·h + h·c). |

**Lo que se ve en la práctica.** Distancia Mínima y Naïve Bayes entrenan y predicen rápido. 1-NN y K-NN son al revés: entrenan al instante pero cada predicción es más lenta mientras más datos haya. SVM y MLP son los que más tardan en entrenar, y por eso el ajuste de parámetros con ellos fue lo que más se tardó en correr (más de un minuto en total, con las búsquedas en rejilla).

# Ajuste de parámetros (K-NN, SVM y MLP)

Probamos parámetros distintos a los de default: varios valores de k en K-NN, y una búsqueda en rejilla (`GridSearchCV` con 5 folds) para el kernel, C y gamma de SVM, y para las capas ocultas, activación y alpha del MLP. La búsqueda se hizo solo con el 70% de entrenamiento y el 30% de prueba se usó al final, para comparar contra los resultados con parámetros por defecto.

**¿Mejoraron los resultados?** En general no mucho:
- **Iris:** K-NN (k=7) y SVM (kernel lineal, C=0.1) quedaron igual que con el default (0.9778). Solo el MLP subió, de 0.9333 a 0.9556. Como Iris ya estaba cerca del máximo, casi no hay margen de mejora.
- **Sintético:** K-NN con k=5 quedó igual (0.7222), SVM bajó un poco (0.7444 a 0.7333) y el MLP bajó bastante (0.7444 a 0.6000).

Creemos que en el sintético no mejoró porque el 15% de etiquetas cambiadas al azar es ruido que ningún parámetro puede arreglar, y al buscar la mejor combinación se elige la que salió mejor en validación por suerte, que luego no se repite en el conjunto de prueba. En el MLP, además, el default se detenía a las 200 iteraciones sin converger, y con `max_iter=2000` el modelo entrenó hasta el final y pudo ajustarse más al ruido. Con solo 90 datos de prueba, unas pocas predicciones mueven mucho el resultado, así que estas diferencias hay que tomarlas con cuidado.

**¿Cómo encontrar los mejores parámetros para cada dataset?** Hay que buscarlos por separado en cada dataset, porque salieron distintos (por ejemplo, SVM prefirió kernel lineal en Iris y RBF en el sintético). Lo más común es:
- **Búsqueda en rejilla (`GridSearchCV`):** prueba todas las combinaciones de una lista de valores usando validación cruzada. Es la que usamos; es sencilla pero se vuelve lenta si hay muchos parámetros.
- **Búsqueda aleatoria (`RandomizedSearchCV`):** prueba combinaciones al azar, y suele ser más rápida con muchos parámetros.
- **Optimización bayesiana:** elige la siguiente combinación según lo que ya probó, y sirve cuando cada entrenamiento es muy caro.

En todos los casos hay que elegir los parámetros con validación cruzada sobre el conjunto de entrenamiento, y reservar un conjunto de prueba que no se toque hasta el final. Lo ideal para estimar el desempeño real es una validación cruzada anidada, que no hicimos por el tiempo de cómputo.

# Conclusión
Para que la comparación fuera justa, a los seis clasificadores les aplicamos la misma partición Hold-Out 70/30 y los mismos folds de K-Fold (k=5), fijando la semilla para que los resultados se puedan repetir.

Con el dataset benchmark (Iris) todos los modelos funcionaron bien, con 0.92 o más en K-Fold. El más flojo fue Distancia Mínima (0.92 en K-Fold y 0.889 en Hold-Out), lo cual tiene sentido porque representa cada clase con un solo punto central y se queda corto cuando las clases no son tan compactas. K-NN (k=3) y SVM quedaron arriba con 0.9667, y 1-NN, Naïve Bayes y MLP quedaron muy cerca (0.96, 0.96 y 0.9467).

Con el dataset sintético con ruido el rendimiento bajó en todos, porque al mezclar etiquetas las clases se traslapan y ya no hay fronteras claras. SVM (0.75), MLP (0.7533) y K-NN con k=3 (0.7233) fueron los que mejor aguantaron; Naïve Bayes (0.6233), 1-NN (0.6567) y Distancia Mínima (0.5667) fueron los más afectados. Algo que nos llamó la atención es que K-NN con k=3 le gana bastante a 1-NN: 1-NN copia la clase de un solo vecino, y si justo ese punto es ruido se equivoca, mientras que con tres vecinos la votación ayuda a ignorar esos casos. Naïve Bayes y Distancia Mínima también batallaron porque asumen cosas muy simples sobre los datos (rasgos independientes y una sola zona por clase) que aquí no se cumplen. Las diferencias entre SVM, MLP y K-NN son de apenas 2 o 3 puntos con 300 muestras, así que más que un ganador claro lo vemos como un empate entre los tres.

También notamos que no conviene fiarse solo del Hold-Out. Por ejemplo, MLP en Iris sacó 0.9333 con Hold-Out y 0.9467 con K-Fold, y Distancia Mínima pasó de 0.889 a 0.92. Con Hold-Out el resultado depende de una sola partición al azar (en Iris son solo 45 datos de prueba, así que cada error pesa mucho), mientras que K-Fold promedia cinco particiones y da un número más estable.

Al probar parámetros distintos a los de default en K-NN, SVM y MLP, los resultados no mejoraron de forma clara: en Iris quedaron igual o subieron un poco (solo el MLP), y en el sintético quedaron igual o incluso bajaron. Eso nos dejó ver que ajustar parámetros no garantiza una mejora, sobre todo cuando el ruido está en los datos y no en el modelo, y que el ajuste siempre hay que validarlo con datos que no se usaron para elegir los parámetros.

Por último, al programarlos desde cero se ve la diferencia de costo: Distancia Mínima predice rápido porque solo compara cada punto contra un centro por clase, mientras que 1-NN y K-NN calculan la distancia contra todo el conjunto de entrenamiento cada vez que clasifican un punto nuevo.